In [2]:
%load_ext autoreload
%autoreload 2

## Download All Actas from Website

In [ ]:
import socket
import pandas as pd
from loguru import logger
from playwright.async_api import async_playwright, Page

from elecc_colombia.config import (RAW_DATA_DIR, INTERIM_DATA_DIR,
                                   EXTERNAL_DATA_DIR, HEADLESS,
                                   SLOW_MO, READY_SELECTOR)
from elecc_colombia.actas_log import save_actas_log
from elecc_colombia.browser_utils import navigate
from elecc_colombia.actas_scraper import download_all_actas

# ── Configuration ─────────────────────────────────────────────────────────────
VUELTA = "vuelta02"   # "vuelta01" or "vuelta02"
HOSTNAME = socket.gethostname().split(".")[0]
LOG_PATH = INTERIM_DATA_DIR / VUELTA / f"actas_{HOSTNAME}_log.csv"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Vuelta : {VUELTA}")
print(f"Computer: {HOSTNAME}")
print(f"Log    : {LOG_PATH}")

## Read List of Departamentos URLs

In [ ]:
deptos_url_df = pd.read_csv(EXTERNAL_DATA_DIR / f"lista_{VUELTA}_actas_urls.csv")
deptos_url_df["DEPARTAMENTO"] = deptos_url_df["DEPARTAMENTO"].str.strip()
deptos_url_df

In [ ]:
list_complete_deptos = ['BOLIVAR']
logger.info(f"--- Reading Results from {VUELTA} ---")

async with async_playwright() as pw:
    browser = await pw.chromium.launch(headless=HEADLESS, slow_mo=SLOW_MO)

    for n_depto, row in deptos_url_df.iterrows():
        departamento = row["DEPARTAMENTO"]
        url = row["URL_ACTAS"]

        if departamento not in list_complete_deptos:
            print(f"Skipping departamento: {departamento}")
            continue

        download_dir = RAW_DATA_DIR / VUELTA / departamento.replace(" ", "_")

        logger.info(f"Starting departamento: {departamento} — {url}")
        page = await browser.new_page()
        try:
            await navigate(page, url, READY_SELECTOR)
            records = await download_all_actas(page, download_dir, departamento=departamento, log_path=LOG_PATH)
            logger.success(f"Finished {departamento} — {len(records)} actas logged")
        except Exception as e:
            logger.error(f"Failed departamento {departamento}: {e}")
        finally:
            await page.close()

        break  # remove to process all departamentos

    await browser.close()

logger.info(f"Log saved to: {LOG_PATH}")